# Sprint 4: Modelo Híbrido CNN-ViT

## Arquitectura
```
Input (B, 1, 224, 224)
    ↓
DenseNet121 features (preentrenado NIH)
    ↓ (B, 1024, 7, 7)
Flatten → 49 patches de dim 1024
    ↓
Vision Transformer (4 bloques, 8 cabezas, dim=512)
  ├── Patch Embedding (1024 → 512)
  ├── [CLS] token + Positional Encoding
  └── 4× TransformerBlock (MHSA + MLP + LayerNorm)
    ↓
[CLS] token → Linear(512, 4)
    ↓
logits (B, 4)
```

## Estrategia de entrenamiento — 2 fases
- **Fase 1** (15 epochs): CNN congelada, solo ViT + head. LR=1e-4.
- **Fase 2** (30 epochs): Todo desbloqueado. LR_cnn=1e-6, LR_vit=1e-5.

## Datos
- Train: **TODAS** las imágenes por paciente (~3,900 imgs vs 739 en Sprint 2).
- Val/Test: 1 imagen/paciente (igual que Sprint 2 → comparación directa).

**Runtime: T4 GPU. Tiempo estimado: ~90-120 min total (2 fases).**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, shutil, time
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ CPU — necesitas T4'}")
print(f"torch: {torch.__version__}")

H5_LOCAL        = '/content/nih_images.h5'
PROCESSED_S4    = '/content/processed_s4'
CLASS_NAMES     = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}


In [ ]:
!pip install -q torchxrayvision scipy h5py
print('deps OK')


## 2. Copiar HDF5 y verificar

In [ ]:
import h5py

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 1_000_000_000):
    size = os.path.getsize(f'{NIH_DIR}/nih_images.h5') / (1024**3)
    print(f'Copiando HDF5 ({size:.1f} GB)...')
    t0 = time.time()
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)
    print(f'✓ Copiado en {(time.time()-t0)/60:.1f} min')

with h5py.File(H5_LOCAL, 'r') as hf:
    print(f'✓ HDF5 local: {len(hf["images"]):,} imágenes')


## 3. Instalar archivos del Sprint 4

In [ ]:
from pathlib import Path


In [ ]:
content = r'''# configs/sprint4.yaml
# Sprint 4 — Modelo híbrido CNN-ViT
# ─────────────────────────────────────────────────────────────────
# Arquitectura: Secuencial CNN (DenseNet121) → Vision Transformer
# Tarea: clasificación multi-clase 4 clases (igual que Sprint 2)
# Clases:
#   0 - No Finding
#   1 - Cardiomegaly
#   2 - Effusion
#   3 - Infiltration
#
# Diferencias clave respecto a Sprint 2:
#   - Arquitectura: DenseNet121 puro → DenseNet121 + ViT propio
#   - Datos train: 1 img/paciente (~739) → TODAS las imgs/paciente (~3,900)
#   - Entrenamiento: 2 fases con LR diferenciado por componente
#   - Splits: data/processed_s4/ (nuevos, con más datos en train)

project:
  name: "tesis-cxr-sprint4-cnn-vit"
  sprint: 4
  seed: 42

paths:
  project_root: "/content/drive/MyDrive/Tesis_CXR"
  data_raw:     "/content/drive/MyDrive/Tesis_CXR/data/raw/nih"
  data_processed: "/content/drive/MyDrive/Tesis_CXR/data/processed_s4"
  experiments:  "/content/drive/MyDrive/Tesis_CXR/experiments_s4"
  checkpoints:  "/content/drive/MyDrive/Tesis_CXR/experiments_s4/checkpoints"
  logs:         "/content/drive/MyDrive/Tesis_CXR/experiments_s4/logs"

dataset:
  name: "nih"
  csv_filename: "Data_Entry_subset_local.csv"
  images_subdir: "images"
  views: ["PA", "AP"]
  negative_class: "No Finding"
  classes:
    - {name: "No Finding",   label: 0, priority: 0}
    - {name: "Cardiomegaly", label: 1, priority: 1}
    - {name: "Effusion",     label: 2, priority: 2}
    - {name: "Infiltration", label: 3, priority: 3}

class_ratio: 2.0  # ratio No Finding : positivas (en train, a nivel IMAGEN)

preprocessing:
  image_size: 224

splits:
  train_ratio: 0.70
  val_ratio:   0.15
  test_ratio:  0.15
  # CLAVE: en train se usan TODAS las imágenes del paciente
  # En val y test se usa 1 imagen/paciente (para evaluación limpia)
  all_images_in_train: true

model:
  backbone: "densenet121-res224-nih"
  num_classes: 4
  pretrained: true

  # Hiperparámetros del Vision Transformer
  vit:
    embedding_dim: 512    # dimensión del espacio de atención
    num_heads:     8      # 512 / 8 = 64 dims por cabeza
    num_layers:    4      # bloques Transformer (ViT-Small: 4, ViT-Base: 12)
    mlp_dim:       1024   # dimensión interna del MLP (2× embedding_dim)
    dropout:       0.1    # regularización

training:
  batch_size: 32
  num_workers: 2
  use_amp: true
  use_class_weights: true

  # ─── Fase 1: CNN congelada, solo ViT + head ───────────────────
  phase1:
    epochs:         15
    learning_rate:  0.0001   # LR para el ViT en fase 1
    weight_decay:   0.0001
    optimizer:      "adam"
    patience:       5        # early stopping por fase

  # ─── Fase 2: Todo desbloqueado, LR diferenciado ──────────────
  phase2:
    epochs:         30
    lr_cnn:         0.000001  # 1e-6 para CNN (muy conservador)
    lr_vit:         0.00001   # 1e-5 para ViT (mayor capacidad de actualización)
    weight_decay:   0.0001
    optimizer:      "adam"
    patience:       8

    # Scheduler para fase 2
    scheduler:
      enabled: true
      type: "ReduceLROnPlateau"
      mode: "min"
      factor: 0.5
      patience: 3
      min_lr: 0.0000001

evaluation:
  metrics: ["auc_macro", "auc_per_class", "accuracy",
            "precision_macro", "sensitivity_macro", "specificity_macro"]
  threshold: 0.5

gradcam:
  # Para el CNN-ViT, Grad-CAM se aplica sobre la última capa CNN
  # (antes del ViT), no sobre el Transformer
  target_layer: "cnn_features.denseblock4"
  num_samples_per_class: 2
'''
p = Path(f'{CODE_DIR}/configs/sprint4.yaml')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p} ({len(content):,} chars)')


In [ ]:
content = r'''"""
src/models/cnn_vit.py

Modelo híbrido CNN-ViT para clasificación de radiografías de tórax.
Arquitectura: Secuencial CNN → ViT (Opción A).

Diseño:
  1. CNN Backbone (DenseNet121 preentrenado en NIH via TorchXRayVision):
     extrae features locales de la imagen CXR.
     Output: feature map (B, 1024, 7, 7).

  2. Patch Embedding:
     Convierte el feature map en una secuencia de patches para el Transformer.
     7×7 = 49 patches, proyectados a embedding_dim=512.

  3. Vision Transformer (implementación propia):
     [CLS] token + positional encoding + N bloques Transformer.
     Cada bloque: LayerNorm → Multi-Head Self-Attention → MLP.
     El [CLS] token agrega información global de todos los patches.

  4. Cabeza clasificadora:
     LayerNorm → Linear(embedding_dim, num_classes).

Justificación del diseño:
  El CNN pre-procesa la imagen en features de dominio (CXR), reduciendo
  la tarea del ViT de "aprender features desde píxeles" a "relacionar
  features ya significativas". Esto reduce drásticamente el volumen de
  datos necesario para entrenar el ViT desde ~14M imágenes (ViT original)
  a ~3,900 imágenes de CXR.

Referencias:
  - Dosovitskiy et al. (2021). An Image is Worth 16x16 Words. ICLR.
  - Chen et al. (2021). Crossvit: Cross-attention multi-scale ViT. ICCV.
  - TorchXRayVision: Cohen et al. (2022).
"""
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torchxrayvision as xrv


# ---------------------------------------------------------------------------
# Transformer components
# ---------------------------------------------------------------------------
class MultiHeadSelfAttention(nn.Module):
    """
    Multi-Head Self-Attention con proyecciones Q, K, V.
    Implementación propia para máxima trazabilidad y comprensión.
    """
    def __init__(self, embedding_dim: int, num_heads: int,
                 dropout: float = 0.0):
        super().__init__()
        assert embedding_dim % num_heads == 0, \
            f"embedding_dim={embedding_dim} no divisible entre num_heads={num_heads}"

        self.num_heads = num_heads
        self.head_dim  = embedding_dim // num_heads
        self.scale     = math.sqrt(self.head_dim)

        self.qkv  = nn.Linear(embedding_dim, embedding_dim * 3, bias=False)
        self.proj = nn.Linear(embedding_dim, embedding_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, N, D) — N tokens de dimensión D.
        Returns:
            (B, N, D)
        """
        B, N, D = x.shape

        # Proyectar a Q, K, V
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)   # (3, B, H, N, d_head)
        q, k, v = qkv.unbind(0)             # cada: (B, H, N, d_head)

        # Atención escalada
        attn = (q @ k.transpose(-2, -1)) / self.scale  # (B, H, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # Agregación ponderada
        out = (attn @ v).transpose(1, 2).reshape(B, N, D)  # (B, N, D)
        return self.proj_drop(self.proj(out))


class TransformerBlock(nn.Module):
    """
    Bloque Transformer estándar:
      LayerNorm → MHSA → residual
      LayerNorm → MLP  → residual
    """
    def __init__(self, embedding_dim: int, num_heads: int,
                 mlp_dim: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.attn  = MultiHeadSelfAttention(embedding_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.mlp   = nn.Sequential(
            nn.Linear(embedding_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embedding_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))   # residual MHSA
        x = x + self.mlp(self.norm2(x))    # residual MLP
        return x


class VisionTransformer(nn.Module):
    """
    Vision Transformer aplicado a patches de features (no píxeles).

    Args:
        n_patches:     número de patches de entrada (49 para 7×7 feature map).
        patch_dim:     dimensión de cada patch (1024 del DenseNet121).
        embedding_dim: dimensión del espacio de atención.
        num_heads:     cabezas de atención.
        num_layers:    bloques Transformer.
        mlp_dim:       dimensión interna del MLP.
        num_classes:   número de clases de salida.
        dropout:       regularización.
    """
    def __init__(self, n_patches: int, patch_dim: int,
                 embedding_dim: int = 512, num_heads: int = 8,
                 num_layers: int = 4, mlp_dim: int = 1024,
                 num_classes: int = 4, dropout: float = 0.1):
        super().__init__()

        self.n_patches     = n_patches
        self.embedding_dim = embedding_dim

        # Proyección de patches → espacio de atención
        self.patch_embedding = nn.Linear(patch_dim, embedding_dim)

        # [CLS] token learnable
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embedding_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Positional encoding learnable (n_patches + 1 por el CLS)
        self.pos_encoding = nn.Parameter(
            torch.zeros(1, n_patches + 1, embedding_dim)
        )
        nn.init.trunc_normal_(self.pos_encoding, std=0.02)

        self.dropout = nn.Dropout(dropout)

        # Bloques Transformer
        self.blocks = nn.ModuleList([
            TransformerBlock(embedding_dim, num_heads, mlp_dim, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(embedding_dim)

        # Cabeza clasificadora
        self.head = nn.Linear(embedding_dim, num_classes)

        self._init_weights()

    def _init_weights(self) -> None:
        """Inicialización estándar para Transformers."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, patches: torch.Tensor) -> torch.Tensor:
        """
        Args:
            patches: (B, n_patches, patch_dim) — patches del feature map CNN.
        Returns:
            logits: (B, num_classes).
        """
        B = patches.shape[0]

        # Proyectar patches al espacio de atención
        x = self.patch_embedding(patches)  # (B, 49, 512)

        # Prepend [CLS] token
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, 512)
        x   = torch.cat([cls, x], dim=1)         # (B, 50, 512)

        # Agregar positional encoding
        x = self.dropout(x + self.pos_encoding)

        # Pasar por todos los bloques Transformer
        for block in self.blocks:
            x = block(x)

        # Tomar el [CLS] token y normalizar
        cls_out = self.norm(x[:, 0])  # (B, 512)

        return self.head(cls_out)     # (B, num_classes)


# ---------------------------------------------------------------------------
# Modelo híbrido CNN-ViT
# ---------------------------------------------------------------------------
class CNNViT(nn.Module):
    """
    Modelo híbrido CNN-ViT para clasificación de radiografías de tórax.

    Flujo:
      imagen (B, 1, 224, 224)
        → DenseNet121 features (B, 1024, 7, 7)
        → flatten spatial: (B, 49, 1024)
        → VisionTransformer: (B, 49, 1024) → (B, num_classes)

    Soporta entrenamiento en 2 fases:
      Fase 1: CNN congelada, solo ViT + head (set_phase(1))
      Fase 2: Todo desbloqueado con LR diferenciado (set_phase(2))
    """

    def __init__(self, backbone_weights: str = "densenet121-res224-nih",
                 embedding_dim: int = 512, num_heads: int = 8,
                 num_layers: int = 4, mlp_dim: int = 1024,
                 num_classes: int = 4, dropout: float = 0.1):
        super().__init__()

        # 1. CNN Backbone preentrenado
        densenet = xrv.models.DenseNet(weights=backbone_weights)
        self.cnn_features = densenet.features
        self.cnn_out_dim  = 1024  # canales del feature map del DenseNet121

        # 2. Vision Transformer
        # 7×7 feature map → 49 patches
        self.vit = VisionTransformer(
            n_patches=49,
            patch_dim=self.cnn_out_dim,
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            num_layers=num_layers,
            mlp_dim=mlp_dim,
            num_classes=num_classes,
            dropout=dropout,
        )

        self._current_phase = None

    def set_phase(self, phase: int) -> None:
        """
        Configura qué parámetros se entrenan.

        Fase 1: CNN congelada, solo ViT + head.
        Fase 2: Todo desbloqueado.
        """
        assert phase in {1, 2}, "phase debe ser 1 o 2"
        self._current_phase = phase

        if phase == 1:
            # Congelar CNN
            for param in self.cnn_features.parameters():
                param.requires_grad = False
            # Descongelar ViT
            for param in self.vit.parameters():
                param.requires_grad = True

        else:  # phase 2
            # Descongelar todo
            for param in self.parameters():
                param.requires_grad = True

    def get_param_groups(self, lr_cnn: float, lr_vit: float) -> list[dict]:
        """
        Devuelve grupos de parámetros con LR diferenciado para la Fase 2.

        CNN recibe lr_cnn (muy bajo para no destruir preentrenamiento).
        ViT recibe lr_vit (más alto para seguir aprendiendo).
        """
        return [
            {"params": self.cnn_features.parameters(), "lr": lr_cnn,
             "name": "cnn_backbone"},
            {"params": self.vit.parameters(), "lr": lr_vit,
             "name": "vit"},
        ]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 1, 224, 224) en rango [-1024, 1024] (formato xrv).
        Returns:
            logits: (B, num_classes).
        """
        # Extraer feature map del DenseNet
        features = self.cnn_features(x)           # (B, 1024, 7, 7)

        # ReLU (igual que el forward original del DenseNet)
        import torch.nn.functional as F
        features = F.relu(features, inplace=False)  # (B, 1024, 7, 7)

        # Convertir feature map en secuencia de patches
        B, C, H, W = features.shape
        patches = features.flatten(2).transpose(1, 2)  # (B, 49, 1024)

        # Pasar por el ViT
        return self.vit(patches)                   # (B, num_classes)

    def trainable_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_params(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def param_summary(self) -> dict:
        cnn_total = sum(p.numel() for p in self.cnn_features.parameters())
        vit_total = sum(p.numel() for p in self.vit.parameters())
        return {
            "cnn_total":   cnn_total,
            "vit_total":   vit_total,
            "total":       cnn_total + vit_total,
            "trainable":   self.trainable_params(),
            "phase":       self._current_phase,
        }


# ---------------------------------------------------------------------------
# Factory
# ---------------------------------------------------------------------------
def build_cnn_vit(cfg) -> CNNViT:
    """Construye el modelo CNN-ViT desde la config del Sprint 4."""
    vit_cfg = cfg.model.get("vit", {})
    model = CNNViT(
        backbone_weights=cfg.model["backbone"],
        embedding_dim=vit_cfg.get("embedding_dim", 512),
        num_heads=vit_cfg.get("num_heads", 8),
        num_layers=vit_cfg.get("num_layers", 4),
        mlp_dim=vit_cfg.get("mlp_dim", 1024),
        num_classes=cfg.model["num_classes"],
        dropout=vit_cfg.get("dropout", 0.1),
    )
    return model


# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import sys
    from pathlib import Path
    code_dir = Path(__file__).resolve().parent.parent.parent
    if str(code_dir) not in sys.path:
        sys.path.insert(0, str(code_dir))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = CNNViT(num_classes=4).to(device)
    model.set_phase(1)
    summary = model.param_summary()
    print(f"Modelo CNN-ViT (fase 1):")
    print(f"  CNN params:      {summary['cnn_total']:>12,}")
    print(f"  ViT params:      {summary['vit_total']:>12,}")
    print(f"  Total:           {summary['total']:>12,}")
    print(f"  Entrenables:     {summary['trainable']:>12,} (solo ViT)")

    model.set_phase(2)
    print(f"  Entrenables (fase 2): {model.trainable_params():>8,} (todo)")

    # Forward dummy
    model.set_phase(1)
    model.eval()
    x = torch.randn(4, 1, 224, 224, device=device) * 500
    with torch.no_grad():
        logits = model(x)
    print(f"\nForward OK: input={x.shape} → output={logits.shape}")
    probs = torch.softmax(logits, dim=1)
    print(f"Probs[0]: {probs[0].tolist()}")
    print(f"Suma: {probs[0].sum().item():.3f}")
'''
p = Path(f'{CODE_DIR}/src/models/cnn_vit.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p} ({len(content):,} chars)')


In [ ]:
content = r'''"""
scripts/prepare_splits_s4.py

Genera los splits para Sprint 4 con una estrategia de datos mejorada:

DIFERENCIA CLAVE respecto a Sprint 2:
  - Train: TODAS las imágenes disponibles de cada paciente seleccionado.
           No se limita a 1 imagen/paciente.
  - Val/Test: 1 imagen/paciente (igual que Sprint 2, para evaluación limpia).

Justificación:
  La regla de 1 imagen/paciente en train fue conservadora para el baseline.
  Para el CNN-ViT necesitamos más datos en train para que el ViT aprenda
  relaciones globales. Val y test mantienen 1 imagen/paciente para garantizar
  que la evaluación sea comparable con Sprint 2.

Resultado esperado:
  Sprint 2: ~739  imágenes train (1/paciente)
  Sprint 4: ~3,900 imágenes train (todas/paciente, estimado)
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.training.utils import set_seed, get_logger, load_config


CLASS_NAMES = {
    0: "No Finding",
    1: "Cardiomegaly",
    2: "Effusion",
    3: "Infiltration",
}

LABEL_HIERARCHY = {
    "Infiltration": 3,
    "Effusion":     2,
    "Cardiomegaly": 1,
    "No Finding":   0,
}


def build_multiclass_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Asigna label por jerarquía clínica (igual que Sprint 2)."""
    df = df.copy()

    def assign(finding: str) -> float:
        for keyword, label in sorted(LABEL_HIERARCHY.items(),
                                     key=lambda x: -x[1]):
            if keyword == "No Finding":
                if finding == "No Finding":
                    return 0
            elif keyword in finding:
                return label
        return float("nan")

    df["label"] = df["Finding Labels"].apply(assign)
    return df


def filter_views(df: pd.DataFrame, views: list) -> pd.DataFrame:
    return df[df["View Position"].isin(views)].copy()


def build_patient_label(df: pd.DataFrame) -> pd.Series:
    """Label representativo por paciente (máximo = prioridad más alta)."""
    return df.groupby("Patient ID")["label"].max().astype(int)


def balance_patients(df: pd.DataFrame, ratio: float, seed: int,
                     logger) -> set:
    """
    Selecciona pacientes balanceados.
    Devuelve el set de Patient IDs seleccionados.
    """
    rng = np.random.default_rng(seed)
    patient_labels = build_patient_label(df)

    patients_by_class = {
        label: list(patient_labels[patient_labels == label].index)
        for label in [0, 1, 2, 3]
    }

    # Techo = mínimo entre clases positivas
    ceiling = min(len(patients_by_class[l]) for l in [1, 2, 3])
    logger.info("Techo de balance: %d pacientes por clase positiva", ceiling)

    selected = set()
    for label in [1, 2, 3]:
        pats = patients_by_class[label]
        rng.shuffle(pats)
        chosen = pats[:ceiling]
        selected.update(chosen)
        logger.info("  Clase %d (%s): %d/%d pacientes",
                    label, CLASS_NAMES[label], len(chosen), len(pats))

    # No Finding
    n_nf = min(len(patients_by_class[0]), int(ceiling * ratio))
    nf_pats = patients_by_class[0]
    rng.shuffle(nf_pats)
    selected.update(nf_pats[:n_nf])
    logger.info("  Clase 0 (No Finding): %d pacientes", n_nf)

    logger.info("Total pacientes seleccionados: %d", len(selected))
    return selected


def stratified_split_patients(df: pd.DataFrame, selected_patients: set,
                               train_ratio: float, val_ratio: float,
                               test_ratio: float, seed: int, logger
                               ) -> tuple[set, set, set]:
    """Split estratificado a nivel paciente."""
    patient_labels = build_patient_label(df)
    patient_labels = patient_labels[patient_labels.index.isin(selected_patients)]

    X = patient_labels.index.to_numpy()
    y = patient_labels.values

    # Separar test
    n_test = max(2, round(1.0 / test_ratio))
    skf1 = StratifiedKFold(n_splits=n_test, shuffle=True, random_state=seed)
    tv_idx, test_idx = next(skf1.split(X, y))

    test_pats = set(X[test_idx])
    X_tv, y_tv = X[tv_idx], y[tv_idx]

    # Separar val de train
    val_rel = val_ratio / (train_ratio + val_ratio)
    n_val = max(2, round(1.0 / val_rel))
    skf2 = StratifiedKFold(n_splits=n_val, shuffle=True, random_state=seed)
    tr_idx, val_idx = next(skf2.split(X_tv, y_tv))

    train_pats = set(X_tv[tr_idx])
    val_pats   = set(X_tv[val_idx])

    logger.info("Split pacientes: train=%d, val=%d, test=%d",
                len(train_pats), len(val_pats), len(test_pats))
    return train_pats, val_pats, test_pats


def sample_one_per_patient(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    """1 imagen aleatoria por paciente (para val y test)."""
    rng = np.random.default_rng(seed)
    rows = []
    for _, group in df.groupby("Patient ID"):
        idx = rng.integers(0, len(group))
        rows.append(group.iloc[idx])
    return pd.DataFrame(rows).reset_index(drop=True)


def report_split(name: str, df: pd.DataFrame, logger) -> None:
    n = len(df)
    n_pat = df["Patient ID"].nunique()
    logger.info("%s: %d imgs | %d pacientes", name.upper(), n, n_pat)
    for label, cname in CLASS_NAMES.items():
        count = (df["label"] == label).sum()
        logger.info("  clase %d (%s): %d (%.1f%%)",
                    label, cname, count, count / n * 100 if n else 0)


def main(config_path=None):
    if config_path is None:
        config_path = CODE_DIR / "configs" / "sprint4.yaml"
    cfg = load_config(config_path)

    set_seed(cfg.project["seed"])
    log_file = Path(cfg.paths["logs"]) / "prepare_splits_s4.log"
    Path(cfg.paths["logs"]).mkdir(parents=True, exist_ok=True)
    logger = get_logger("prepare_splits_s4", log_file=log_file)

    logger.info("=" * 65)
    logger.info("PREPARE SPLITS Sprint 4 — Todas las imágenes en train")
    logger.info("=" * 65)

    # 1. Cargar CSV
    subset_csv = Path(cfg.paths["data_raw"]) / "Data_Entry_subset_local.csv"
    df = pd.read_csv(subset_csv)
    logger.info("CSV cargado: %d filas", len(df))

    # 2. Filtrar vistas y construir labels
    df = filter_views(df, cfg.dataset["views"])
    df = build_multiclass_labels(df)
    n_before = len(df)
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)
    logger.info("Tras filtrar y etiquetar: %d filas (%d descartadas)",
                len(df), n_before - len(df))

    for label, cname in CLASS_NAMES.items():
        logger.info("  %s: %d imágenes, %d pacientes",
                    cname,
                    (df["label"] == label).sum(),
                    df[df["label"] == label]["Patient ID"].nunique())

    # 3. Seleccionar pacientes balanceados
    ratio = cfg.get("class_ratio", 2.0)
    selected_patients = balance_patients(
        df, ratio=ratio, seed=cfg.project["seed"], logger=logger
    )
    df_balanced = df[df["Patient ID"].isin(selected_patients)].copy()

    # 4. Split estratificado por paciente
    processed_dir = Path(cfg.paths["data_processed"])
    processed_dir.mkdir(parents=True, exist_ok=True)

    train_pats, val_pats, test_pats = stratified_split_patients(
        df_balanced, selected_patients,
        cfg.splits["train_ratio"],
        cfg.splits["val_ratio"],
        cfg.splits["test_ratio"],
        seed=cfg.project["seed"],
        logger=logger,
    )

    # Verificar no leakage
    assert not (train_pats & val_pats), "Leakage train-val"
    assert not (train_pats & test_pats), "Leakage train-test"
    assert not (val_pats & test_pats), "Leakage val-test"
    logger.info("Sin leakage entre splits.")

    # 5. Construir DataFrames
    # TRAIN: TODAS las imágenes de los pacientes seleccionados
    df_train = df_balanced[df_balanced["Patient ID"].isin(train_pats)].reset_index(drop=True)

    # VAL/TEST: 1 imagen por paciente
    df_val_full  = df_balanced[df_balanced["Patient ID"].isin(val_pats)]
    df_test_full = df_balanced[df_balanced["Patient ID"].isin(test_pats)]
    df_val  = sample_one_per_patient(df_val_full,  seed=cfg.project["seed"] + 1)
    df_test = sample_one_per_patient(df_test_full, seed=cfg.project["seed"] + 2)

    # 6. Reportar
    logger.info("-" * 65)
    report_split("TRAIN (todas las imágenes/paciente)", df_train, logger)
    report_split("VAL   (1 imagen/paciente)", df_val, logger)
    report_split("TEST  (1 imagen/paciente)", df_test, logger)

    # 7. Guardar
    cols = ["Image Index", "Finding Labels", "Patient ID", "View Position", "label"]
    df_train[cols].to_csv(processed_dir / "train.csv", index=False)
    df_val[cols].to_csv(processed_dir / "val.csv", index=False)
    df_test[cols].to_csv(processed_dir / "test.csv", index=False)

    logger.info("-" * 65)
    logger.info("Splits guardados en: %s", processed_dir)
    logger.info("  train.csv: %d filas", len(df_train))
    logger.info("  val.csv:   %d filas", len(df_val))
    logger.info("  test.csv:  %d filas", len(df_test))
    logger.info("  Ratio train imgs/train S2: %.1fx",
                len(df_train) / 739)
    logger.info("=" * 65)
    logger.info("PREPARE SPLITS S4 OK")


if __name__ == "__main__":
    main()
'''
p = Path(f'{CODE_DIR}/scripts/prepare_splits_s4.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p} ({len(content):,} chars)')


In [ ]:
content = r'''"""
src/training/train_s4.py

Loop de entrenamiento Sprint 4 — CNN-ViT híbrido, 2 fases.

Fase 1 (CNN congelada):
  - Solo se entrenan los parámetros del ViT y la cabeza clasificadora.
  - LR alto (1e-4): el ViT aprende rápido a usar las features de la CNN.
  - Epochs: 15, patience: 5.

Fase 2 (full fine-tuning con LR diferenciado):
  - CNN: lr_cnn=1e-6 (muy conservador, no destruir preentrenamiento).
  - ViT: lr_vit=1e-5 (sigue aprendiendo pero más despacio que en fase 1).
  - Epochs: 30, patience: 8.

El checkpoint final guarda el mejor modelo de la FASE 2.
El checkpoint de fase 1 se guarda por separado (útil para ablation study).
"""
from __future__ import annotations

import copy
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader

from src.evaluation.metrics_s2 import (
    compute_metrics_multiclass,
    logits_to_probs_multiclass,
)
from src.training.utils import save_checkpoint


# ---------------------------------------------------------------------------
# Calcular class weights
# ---------------------------------------------------------------------------
def compute_class_weights(loader: DataLoader, num_classes: int,
                          device: torch.device) -> torch.Tensor:
    counts = torch.zeros(num_classes)
    for batch in loader:
        for label in batch["label"]:
            counts[label.item()] += 1
    total   = counts.sum()
    weights = total / (num_classes * counts)
    return (weights / weights.sum() * num_classes).to(device)


# ---------------------------------------------------------------------------
# Una epoch de entrenamiento
# ---------------------------------------------------------------------------
def train_one_epoch(model: nn.Module, loader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    criterion: nn.Module,
                    device: torch.device,
                    use_amp: bool,
                    scaler) -> dict:
    model.train()
    losses, all_probs, all_labels = [], [], []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp and scaler is not None:
            with autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        losses.append(loss.item())
        with torch.no_grad():
            probs = logits_to_probs_multiclass(logits.detach().float())
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    all_probs  = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return {
        "loss":    float(np.mean(losses)),
        "metrics": compute_metrics_multiclass(all_labels, all_probs),
    }


# ---------------------------------------------------------------------------
# Una epoch de validación
# ---------------------------------------------------------------------------
@torch.no_grad()
def validate_one_epoch(model: nn.Module, loader: DataLoader,
                       criterion: nn.Module,
                       device: torch.device) -> dict:
    model.eval()
    losses, all_probs, all_labels = [], [], []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        logits = model(images)
        losses.append(criterion(logits, labels).item())
        probs = logits_to_probs_multiclass(logits.float())
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

    all_probs  = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return {
        "loss":    float(np.mean(losses)),
        "metrics": compute_metrics_multiclass(all_labels, all_probs),
    }


# ---------------------------------------------------------------------------
# Loop de una fase
# ---------------------------------------------------------------------------
def run_phase(model: nn.Module, train_loader: DataLoader,
              val_loader: DataLoader, optimizer: torch.optim.Optimizer,
              criterion: nn.Module, scheduler, epochs: int,
              patience: int, device: torch.device,
              use_amp: bool, logger, phase_name: str,
              ckpt_path: Path) -> tuple[dict, list]:
    """
    Corre una fase de entrenamiento completa.
    Returns: (mejor estado del modelo, historial de epochs)
    """
    scaler = GradScaler(device="cuda") if use_amp else None

    best_auc   = -float("inf")
    best_state = None
    no_improve = 0
    history    = []

    logger.info("─" * 65)
    logger.info("FASE %s | %d epochs | patience=%d", phase_name, epochs, patience)
    logger.info("Parámetros entrenables: %d", model.trainable_params())
    logger.info("─" * 65)

    for epoch in range(1, epochs + 1):
        lr = optimizer.param_groups[0]["lr"]
        tr = train_one_epoch(model, train_loader, optimizer,
                             criterion, device, use_amp, scaler)
        va = validate_one_epoch(model, val_loader, criterion, device)

        if scheduler is not None:
            scheduler.step(va["loss"])

        logger.info(
            "[%s] Epoch %2d/%d | LR=%.2e | "
            "train loss=%.4f AUC=%.4f | val loss=%.4f AUC=%.4f acc=%.4f",
            phase_name, epoch, epochs, lr,
            tr["loss"], tr["metrics"]["auc_macro"],
            va["loss"], va["metrics"]["auc_macro"], va["metrics"]["accuracy"],
        )
        for i, auc in enumerate(va["metrics"]["auc_per_class"]):
            logger.info("  val AUC clase %d: %.4f", i, auc)

        history.append({
            "phase": phase_name, "epoch": epoch, "lr": lr,
            "train_loss": tr["loss"], "train_metrics": tr["metrics"],
            "val_loss":   va["loss"], "val_metrics":   va["metrics"],
        })

        score = va["metrics"]["auc_macro"]
        if score > best_auc:
            best_auc   = score
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            logger.info("  → Nuevo mejor (val AUC_macro=%.4f)", best_auc)
            # Guardar checkpoint de esta fase
            save_checkpoint({
                "phase": phase_name, "epoch": epoch,
                "model_state_dict": best_state,
                "best_val_auc_macro": best_auc,
            }, ckpt_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                logger.info("  Early stopping en %s (epoch %d).", phase_name, epoch)
                break

    logger.info("Fase %s terminada. Mejor AUC_macro=%.4f", phase_name, best_auc)
    return best_state, history


# ---------------------------------------------------------------------------
# Loop principal — 2 fases
# ---------------------------------------------------------------------------
def train_model_cnn_vit(model: nn.Module,
                        train_loader: DataLoader,
                        val_loader: DataLoader,
                        cfg: Any,
                        device: torch.device,
                        logger) -> dict:
    """
    Entrena el CNN-ViT en 2 fases.

    Fase 1: CNN congelada, solo ViT + head.
    Fase 2: Todo desbloqueado, LR diferenciado CNN vs ViT.
    """
    ckpt_dir = Path(cfg.paths["checkpoints"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    use_amp = cfg.training.get("use_amp", False) and torch.cuda.is_available()
    use_cw  = cfg.training.get("use_class_weights", True)

    # Class weights (calculados una vez)
    if use_cw:
        logger.info("Calculando class weights...")
        weights   = compute_class_weights(train_loader, cfg.model["num_classes"], device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        logger.info("Class weights: %s", weights.tolist())
    else:
        criterion = nn.CrossEntropyLoss()

    p1_cfg = cfg.training["phase1"]
    p2_cfg = cfg.training["phase2"]
    all_history = []

    # ─── FASE 1 ──────────────────────────────────────────────────
    logger.info("=" * 65)
    logger.info("INICIANDO FASE 1: CNN congelada, entrenando solo ViT")
    logger.info("=" * 65)

    model.set_phase(1)
    logger.info("Arquitectura fase 1: %s", model.param_summary())

    opt1 = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=p1_cfg["learning_rate"],
        weight_decay=p1_cfg.get("weight_decay", 1e-4),
    )

    best_state_p1, history_p1 = run_phase(
        model, train_loader, val_loader, opt1, criterion,
        scheduler=None,
        epochs=p1_cfg["epochs"],
        patience=p1_cfg.get("patience", 5),
        device=device, use_amp=use_amp, logger=logger,
        phase_name="P1",
        ckpt_path=ckpt_dir / "sprint4_phase1_best.pt",
    )
    all_history.extend(history_p1)

    # Cargar mejor estado de fase 1 antes de iniciar fase 2
    if best_state_p1 is not None:
        model.load_state_dict(best_state_p1)
    best_p1_auc = max(h["val_metrics"]["auc_macro"] for h in history_p1)

    # ─── FASE 2 ──────────────────────────────────────────────────
    logger.info("=" * 65)
    logger.info("INICIANDO FASE 2: full fine-tuning con LR diferenciado")
    logger.info("  CNN LR: %.2e | ViT LR: %.2e",
                p2_cfg["lr_cnn"], p2_cfg["lr_vit"])
    logger.info("=" * 65)

    model.set_phase(2)
    logger.info("Arquitectura fase 2: %s", model.param_summary())

    param_groups = model.get_param_groups(
        lr_cnn=p2_cfg["lr_cnn"],
        lr_vit=p2_cfg["lr_vit"],
    )
    opt2 = torch.optim.Adam(param_groups,
                             weight_decay=p2_cfg.get("weight_decay", 1e-4))

    sched_cfg = p2_cfg.get("scheduler", {})
    scheduler2 = None
    if sched_cfg.get("enabled", False):
        scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt2,
            mode=sched_cfg.get("mode", "min"),
            factor=sched_cfg.get("factor", 0.5),
            patience=sched_cfg.get("patience", 3),
            min_lr=sched_cfg.get("min_lr", 1e-7),
        )

    best_state_p2, history_p2 = run_phase(
        model, train_loader, val_loader, opt2, criterion,
        scheduler=scheduler2,
        epochs=p2_cfg["epochs"],
        patience=p2_cfg.get("patience", 8),
        device=device, use_amp=use_amp, logger=logger,
        phase_name="P2",
        ckpt_path=ckpt_dir / "sprint4_phase2_best.pt",
    )
    all_history.extend(history_p2)

    best_p2_auc = max(h["val_metrics"]["auc_macro"] for h in history_p2)
    best_p2_epoch = next(
        h["epoch"] for h in reversed(history_p2)
        if h["val_metrics"]["auc_macro"] == best_p2_auc
    )

    logger.info("=" * 65)
    logger.info("ENTRENAMIENTO CNN-ViT TERMINADO")
    logger.info("  Fase 1 best AUC_macro: %.4f", best_p1_auc)
    logger.info("  Fase 2 best AUC_macro: %.4f", best_p2_auc)
    logger.info("  Checkpoint final: %s", ckpt_dir / "sprint4_phase2_best.pt")
    logger.info("=" * 65)

    return {
        "history":            all_history,
        "best_epoch_phase1":  max(range(len(history_p1)),
                                   key=lambda i: history_p1[i]["val_metrics"]["auc_macro"]) + 1,
        "best_epoch_phase2":  best_p2_epoch,
        "best_val_auc_phase1": best_p1_auc,
        "best_val_auc_phase2": best_p2_auc,
        "checkpoint_path":    str(ckpt_dir / "sprint4_phase2_best.pt"),
    }
'''
p = Path(f'{CODE_DIR}/src/training/train_s4.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p} ({len(content):,} chars)')


In [ ]:
content = r'''"""
scripts/run_train_s4.py
Entrypoint Sprint 4: CNN-ViT híbrido, 2 fases.
"""
from __future__ import annotations
import json, os, sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih_hdf5 import NIHDatasetHDF5
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.models.cnn_vit import build_cnn_vit
from src.training.train_s4 import train_model_cnn_vit
from src.training.utils import get_device, get_logger, load_config, set_seed


def main():
    h5_path       = os.environ.get("H5_PATH",       "/content/nih_images.h5")
    processed_dir = os.environ.get("PROCESSED_DIR", "/content/processed_s4")
    cfg    = load_config(CODE_DIR / "configs" / "sprint4.yaml")
    set_seed(cfg.project["seed"])
    Path(cfg.paths["logs"]).mkdir(parents=True, exist_ok=True)
    logger = get_logger("train_s4",
                        log_file=Path(cfg.paths["logs"]) / "train_s4.log")
    logger.info("=" * 70)
    logger.info("RUN TRAIN Sprint 4 — CNN-ViT Híbrido | 2 fases")
    logger.info("  h5:        %s", h5_path)
    logger.info("  processed: %s", processed_dir)
    logger.info("=" * 70)
    device = get_device()
    logger.info("Device: %s%s", device,
                f" | {torch.cuda.get_device_name(0)}" if device.type=="cuda" else "")

    train_tfm = build_train_transform(cfg.preprocessing["image_size"])
    eval_tfm  = build_eval_transform(cfg.preprocessing["image_size"])

    train_ds = NIHDatasetHDF5(f"{processed_dir}/train.csv", h5_path, train_tfm)
    val_ds   = NIHDatasetHDF5(f"{processed_dir}/val.csv",   h5_path, eval_tfm)
    test_ds  = NIHDatasetHDF5(f"{processed_dir}/test.csv",  h5_path, eval_tfm)
    logger.info("Train: %d | Val: %d | Test: %d",
                len(train_ds), len(val_ds), len(test_ds))
    logger.info("Distribución train: %s", train_ds.class_distribution())

    nw = min(cfg.training.get("num_workers", 2), 4)
    train_loader = DataLoader(train_ds, batch_size=cfg.training["batch_size"],
                              shuffle=True, num_workers=nw,
                              pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=cfg.training["batch_size"],
                              shuffle=False, num_workers=nw,
                              pin_memory=(device.type=="cuda"))

    model = build_cnn_vit(cfg).to(device)
    summary = model.param_summary()
    logger.info("Modelo CNN-ViT:")
    logger.info("  CNN params:  %d", summary["cnn_total"])
    logger.info("  ViT params:  %d", summary["vit_total"])
    logger.info("  Total:       %d", summary["total"])

    result = train_model_cnn_vit(model, train_loader, val_loader,
                                  cfg, device, logger)

    hp = Path(cfg.paths["experiments"]) / "history_s4.json"
    hp.parent.mkdir(parents=True, exist_ok=True)
    with hp.open("w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    logger.info("History: %s", hp)
    logger.info("RESUMEN FINAL:")
    logger.info("  Fase 1 best AUC: %.4f", result["best_val_auc_phase1"])
    logger.info("  Fase 2 best AUC: %.4f", result["best_val_auc_phase2"])


if __name__ == "__main__":
    main()
'''
p = Path(f'{CODE_DIR}/scripts/run_train_s4.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p} ({len(content):,} chars)')


In [ ]:
content = r'''"""
scripts/run_train_s4.py
Entrypoint Sprint 4: CNN-ViT híbrido, 2 fases.
"""
from __future__ import annotations
import json, os, sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih_hdf5 import NIHDatasetHDF5
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.models.cnn_vit import build_cnn_vit
from src.training.train_s4 import train_model_cnn_vit
from src.training.utils import get_device, get_logger, load_config, set_seed


def main():
    h5_path       = os.environ.get("H5_PATH",       "/content/nih_images.h5")
    processed_dir = os.environ.get("PROCESSED_DIR", "/content/processed_s4")
    cfg    = load_config(CODE_DIR / "configs" / "sprint4.yaml")
    set_seed(cfg.project["seed"])
    Path(cfg.paths["logs"]).mkdir(parents=True, exist_ok=True)
    logger = get_logger("train_s4",
                        log_file=Path(cfg.paths["logs"]) / "train_s4.log")
    logger.info("=" * 70)
    logger.info("RUN TRAIN Sprint 4 — CNN-ViT Híbrido | 2 fases")
    logger.info("  h5:        %s", h5_path)
    logger.info("  processed: %s", processed_dir)
    logger.info("=" * 70)
    device = get_device()
    logger.info("Device: %s%s", device,
                f" | {torch.cuda.get_device_name(0)}" if device.type=="cuda" else "")

    train_tfm = build_train_transform(cfg.preprocessing["image_size"])
    eval_tfm  = build_eval_transform(cfg.preprocessing["image_size"])

    train_ds = NIHDatasetHDF5(f"{processed_dir}/train.csv", h5_path, train_tfm)
    val_ds   = NIHDatasetHDF5(f"{processed_dir}/val.csv",   h5_path, eval_tfm)
    test_ds  = NIHDatasetHDF5(f"{processed_dir}/test.csv",  h5_path, eval_tfm)
    logger.info("Train: %d | Val: %d | Test: %d",
                len(train_ds), len(val_ds), len(test_ds))
    logger.info("Distribución train: %s", train_ds.class_distribution())

    nw = min(cfg.training.get("num_workers", 2), 4)
    train_loader = DataLoader(train_ds, batch_size=cfg.training["batch_size"],
                              shuffle=True, num_workers=nw,
                              pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=cfg.training["batch_size"],
                              shuffle=False, num_workers=nw,
                              pin_memory=(device.type=="cuda"))

    model = build_cnn_vit(cfg).to(device)
    summary = model.param_summary()
    logger.info("Modelo CNN-ViT:")
    logger.info("  CNN params:  %d", summary["cnn_total"])
    logger.info("  ViT params:  %d", summary["vit_total"])
    logger.info("  Total:       %d", summary["total"])

    result = train_model_cnn_vit(model, train_loader, val_loader,
                                  cfg, device, logger)

    hp = Path(cfg.paths["experiments"]) / "history_s4.json"
    hp.parent.mkdir(parents=True, exist_ok=True)
    with hp.open("w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    logger.info("History: %s", hp)
    logger.info("RESUMEN FINAL:")
    logger.info("  Fase 1 best AUC: %.4f", result["best_val_auc_phase1"])
    logger.info("  Fase 2 best AUC: %.4f", result["best_val_auc_phase2"])


if __name__ == "__main__":
    main()
'''
p = Path(f'{CODE_DIR}/scripts/run_train_s4.py')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


## 4. Generar splits Sprint 4

In [ ]:
import subprocess

result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/prepare_splits_s4.py'],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr[-3000:] if len(result.stderr) > 3000 else result.stderr)


## 5. Copiar CSVs a disco local y verificar

In [ ]:
import pandas as pd

os.makedirs(PROCESSED_S4, exist_ok=True)

for csv in ['train.csv', 'val.csv', 'test.csv']:
    shutil.copy(f'{PROJECT_ROOT}/data/processed_s4/{csv}',
                f'{PROCESSED_S4}/{csv}')
    df = pd.read_csv(f'{PROCESSED_S4}/{csv}')
    dist = {CLASS_NAMES[l]: int((df['label']==l).sum()) for l in range(4)}
    print(f'{csv}: {len(df)} imgs | {dist}')

# Comparar con Sprint 2
df_s2_train = pd.read_csv(f'{PROJECT_ROOT}/data/processed_s2/train.csv')
df_s4_train = pd.read_csv(f'{PROCESSED_S4}/train.csv')
print(f'\nComparativa train:')
print(f'  Sprint 2: {len(df_s2_train)} imgs (1/paciente)')
print(f'  Sprint 4: {len(df_s4_train)} imgs (todas/paciente)')
print(f'  Ratio:    {len(df_s4_train)/len(df_s2_train):.1f}x más datos en train')


## 6. Smoke test: verificar arquitectura CNN-ViT

In [ ]:
# Limpiar cache
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

from src.models.cnn_vit import build_cnn_vit
from src.training.utils import load_config, get_device

cfg    = load_config(f'{CODE_DIR}/configs/sprint4.yaml')
device = get_device()

model = build_cnn_vit(cfg).to(device)
model.set_phase(1)
summary = model.param_summary()

print('Arquitectura CNN-ViT:')
print(f'  CNN params (DenseNet121):  {summary["cnn_total"]:>12,}')
print(f'  ViT params (implementación propia): {summary["vit_total"]:>12,}')
print(f'  Total:                     {summary["total"]:>12,}')
print(f'  Entrenables (fase 1):      {summary["trainable"]:>12,} (solo ViT)')
model.set_phase(2)
print(f'  Entrenables (fase 2):      {model.trainable_params():>12,} (todo)')

# Forward dummy
model.set_phase(1); model.eval()
x = torch.randn(4, 1, 224, 224, device=device) * 500
with torch.no_grad():
    logits = model(x)
probs = torch.softmax(logits, dim=1)
print(f'\nForward OK: input={x.shape} → logits={logits.shape}')
print(f'Probs[0]: {[f"{p:.3f}" for p in probs[0].tolist()]}')
print(f'Suma: {probs[0].sum().item():.3f}')
print('✓ Arquitectura CNN-ViT funcionando.')


## 7. Entrenar CNN-ViT — 2 fases

**Fase 1:** ~20-30 min (CNN congelada, solo ViT)
**Fase 2:** ~60-90 min (full fine-tuning)
**Total estimado: ~90-120 min**

Las métricas aparecen epoch por epoch en el log.


In [ ]:
import subprocess

# Crear carpetas de experiments_s4 en Drive
os.makedirs(f'{PROJECT_ROOT}/experiments_s4/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/experiments_s4/logs', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/experiments_s4/figures', exist_ok=True)

env = {**os.environ, 'H5_PATH': H5_LOCAL, 'PROCESSED_DIR': PROCESSED_S4}

print('Iniciando entrenamiento CNN-ViT (2 fases)...')
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_train_s4.py'],
    capture_output=True, text=True, env=env,
)
print('LOG:')
lines = result.stderr.strip().split('\n') if result.stderr else []
print('\n'.join(lines[-100:]))
print(f'\nReturn code: {result.returncode}')


## 8. Curvas y comparativa Sprint 2 vs Sprint 4

In [ ]:
import json, matplotlib.pyplot as plt
import numpy as np

h4_path = f'{PROJECT_ROOT}/experiments_s4/history_s4.json'
h2_path = f'{PROJECT_ROOT}/experiments_s2/history_s2.json'

if not os.path.exists(h4_path):
    print('history_s4.json no encontrado.')
else:
    with open(h4_path) as f: d4 = json.load(f)
    with open(h2_path) as f: d2 = json.load(f)

    hist4 = d4['history']
    hist2 = d2['history']

    print('RESULTADOS SPRINT 4 (val):')
    print(f"  Fase 1 best AUC_macro: {d4['best_val_auc_phase1']:.4f}")
    print(f"  Fase 2 best AUC_macro: {d4['best_val_auc_phase2']:.4f}")
    print()

    # Mejor epoch de cada fase en S4
    p1_hist = [h for h in hist4 if h['phase'] == 'P1']
    p2_hist = [h for h in hist4 if h['phase'] == 'P2']

    best_p2 = max(p2_hist, key=lambda h: h['val_metrics']['auc_macro'])
    print('Val AUC por clase (Sprint 4, Fase 2):')
    for i, auc in enumerate(best_p2['val_metrics']['auc_per_class']):
        sens = best_p2['val_metrics']['sensitivity_per_class'][i]
        print(f"  {CLASS_NAMES[i]:<15}: AUC={auc:.4f} | Sens={sens:.4f}")

    # Sprint 2 best
    best_s2 = d2['history'][d2['best_epoch']-1]
    print()
    print('COMPARATIVA S2 vs S4 (val):')
    print(f"{'Métrica':<20} {'Sprint 2':>10} {'Sprint 4':>10} {'Delta':>10}")
    print('-' * 55)
    for key, label in [
        ('auc_macro', 'AUC macro'), ('accuracy', 'Accuracy')
    ]:
        v2 = best_s2['val_metrics'][key]
        v4 = best_p2['val_metrics'][key]
        delta = v4 - v2
        arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
        print(f"{label:<20} {v2:>10.4f} {v4:>10.4f} {delta:>+10.4f} {arrow}")
    print()
    for i in range(4):
        a2 = best_s2['val_metrics']['auc_per_class'][i]
        a4 = best_p2['val_metrics']['auc_per_class'][i]
        delta = a4 - a2
        arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
        print(f"  {CLASS_NAMES[i]:<15}: S2={a2:.4f} S4={a4:.4f} {delta:>+.4f} {arrow}")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors4 = ['#2196F3','#F44336','#4CAF50','#FF9800']

    # AUC macro: S2 vs S4
    ax = axes[0]
    ep2 = [h['epoch'] for h in hist2]
    auc2 = [h['val_metrics']['auc_macro'] for h in hist2]
    ep4_p1 = [h['epoch'] for h in p1_hist]
    auc4_p1 = [h['val_metrics']['auc_macro'] for h in p1_hist]
    ep4_p2 = [h['epoch'] for h in p2_hist]
    auc4_p2 = [h['val_metrics']['auc_macro'] for h in p2_hist]

    ax.plot(ep2, auc2, 's-', ms=3, label=f'Sprint 2 (best={max(auc2):.4f})', color='coral')
    ax.plot(ep4_p1, auc4_p1, 'o--', ms=3, label=f'S4 Fase1 (best={max(auc4_p1):.4f})', color='steelblue', alpha=0.6)
    ax.plot(ep4_p2, auc4_p2, 'o-',  ms=3, label=f'S4 Fase2 (best={max(auc4_p2):.4f})', color='steelblue')
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax.set_title('Val AUC Macro: Sprint 2 vs CNN-ViT')
    ax.set_xlabel('Epoch'); ax.set_ylabel('AUC macro')
    ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)

    # AUC por clase en S4 Fase 2
    ax2 = axes[1]
    for i in range(4):
        aucs = [h['val_metrics']['auc_per_class'][i] for h in p2_hist]
        ax2.plot(ep4_p2, aucs, 'o-', ms=3, label=CLASS_NAMES[i], color=colors4[i])
    ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax2.set_title('CNN-ViT: Val AUC por clase (Fase 2)')
    ax2.set_xlabel('Epoch'); ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3); ax2.set_ylim(0, 1.05)

    plt.suptitle(f"Sprint 4 CNN-ViT | S2={max(auc2):.4f} → S4={d4['best_val_auc_phase2']:.4f}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{PROJECT_ROOT}/experiments_s4/figures/curves_s4.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('✓ Gráfica guardada.')


## 9. Siguiente paso

Cuando el entrenamiento termine, vuelve al chat para recibir:
1. `evaluate_s4.py` — evaluación sobre test con el checkpoint de Fase 2.
2. Grad-CAM del CNN-ViT (sobre la capa CNN antes del ViT).
3. Tabla comparativa final Sprint 1 vs Sprint 2 vs Sprint 4.
4. Documentación del Sprint 4.
